In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer, AutoModelForMultipleChoice,
    TrainingArguments, Trainer
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
# Fix torchao/peft version conflict
!pip install -q -U torchao

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
options = ['A', 'B', 'C', 'D', 'E']
label_map = {opt: i for i, opt in enumerate(options)}

# ---------------------------------------------------------
# Q1: Label encoding
# ---------------------------------------------------------
train['label'] = train['answer'].map(label_map)
q1_answer = train.loc[150, 'label']
print("Q1 - Encoded label at index 150:", q1_answer)

# ---------------------------------------------------------
# Q2: Prompt-Option formatting (row 0, Option B)
# ---------------------------------------------------------
row0 = train.iloc[0]
option_b_input = str(row0['prompt']) + " [SEP] " + str(row0['B'])
q2_answer = len(option_b_input)
print("Q2 - Length of formatted Option B input:", q2_answer)

# ---------------------------------------------------------
# Setup tokenizer / model (shared for Q3-Q7)
# ---------------------------------------------------------
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def format_row_choices(row):
    return [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]

# ---------------------------------------------------------
# Q3: Single-row MCQ tokenization (row 0)
# ---------------------------------------------------------
choices_0 = format_row_choices(row0)
enc_0 = tokenizer(
    choices_0,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
input_ids_0 = enc_0['input_ids'].unsqueeze(0)  # reshape -> [1, 5, 128]
print("Q3 - input_ids shape:", input_ids_0.shape, "| second dim:", input_ids_0.shape[1])

# ---------------------------------------------------------
# Q4: Batch MCQ tokenization (first 16 rows)
# ---------------------------------------------------------
batch_rows = train.iloc[:16]
all_choices = []
for _, r in batch_rows.iterrows():
    all_choices.extend(format_row_choices(r))

enc_batch = tokenizer(
    all_choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
batch_input_ids = enc_batch['input_ids'].view(16, 5, 128)
total_token_positions = batch_input_ids.numel()
print("Q4 - Batch input_ids shape:", batch_input_ids.shape, "| total token positions:", total_token_positions)

# ---------------------------------------------------------
# Q5 & Q6: Model logits + supervised loss (row 0)
# ---------------------------------------------------------
model = AutoModelForMultipleChoice.from_pretrained(model_name)

attn_0 = enc_0['attention_mask'].unsqueeze(0)
labels_0 = torch.tensor([int(row0['label'])])

with torch.no_grad():
    outputs = model(input_ids=input_ids_0, attention_mask=attn_0, labels=labels_0)

q5_answer = outputs.logits.shape[-1]
q6_answer = outputs.loss.dim()
print("Q5 - Logits shape:", outputs.logits.shape, "| logits per question:", q5_answer)
print("Q6 - Loss tensor dimensions:", q6_answer, "| loss value:", outputs.loss.item())

# ---------------------------------------------------------
# Q7: LoRA trainable parameters
# ---------------------------------------------------------
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
lora_model = get_peft_model(model, lora_config)
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print("Q7 - Trainable parameters:", trainable_params)

# ---------------------------------------------------------
# Q8: HF Dataset preparation (first 100 rows)
# ---------------------------------------------------------
def tokenize_row(row):
    choices = format_row_choices(row)
    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="np"
    )
    return {
        "input_ids": enc["input_ids"],           # shape [5, 128]
        "attention_mask": enc["attention_mask"],  # shape [5, 128]
        "labels": int(row["label"])
    }

subset_100 = train.iloc[:100].reset_index(drop=True)
processed = [tokenize_row(r) for _, r in subset_100.iterrows()]
hf_dataset = Dataset.from_list(processed)
hf_dataset.set_format(type="torch")

first_item_shape = hf_dataset[0]['input_ids'].shape
print("Q8 - First item input_ids shape:", first_item_shape, "| num choices:", first_item_shape[0])

# ---------------------------------------------------------
# Q9: Tiny LoRA fine-tuning (first 32 rows, max_length=64)
# ---------------------------------------------------------
def tokenize_row_64(row):
    choices = format_row_choices(row)
    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="np"
    )
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": int(row["label"])
    }

subset_32 = train.iloc[:32].reset_index(drop=True)
processed_32 = [tokenize_row_64(r) for _, r in subset_32.iterrows()]
train_dataset_32 = Dataset.from_list(processed_32)
train_dataset_32.set_format(type="torch")

# Fresh LoRA model for fine-tuning
base_model = AutoModelForMultipleChoice.from_pretrained(model_name)
ft_lora_model = get_peft_model(base_model, lora_config)

training_args = TrainingArguments(
    output_dir="/kaggle/working/lora_mcq",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=ft_lora_model,
    args=training_args,
    train_dataset=train_dataset_32
)

train_result = trainer.train()
q9_answer = trainer.state.global_step
print("Q9 - Final global_step:", q9_answer)

# ---------------------------------------------------------
# Q10: Probability for Option E after fine-tuning (row 0)
# ---------------------------------------------------------
ft_lora_model.eval()
with torch.no_grad():
    ft_outputs = ft_lora_model(input_ids=input_ids_0, attention_mask=attn_0)
    probs = torch.softmax(ft_outputs.logits, dim=-1)

q10_answer = round(probs[0, 4].item(), 4)  # index 4 = Option E
print("Q10 - Probability of Option E:", q10_answer)

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------
print("\n--- SUMMARY ---")
print(f"Q1: {q1_answer}")
print(f"Q2: {q2_answer}")
print(f"Q3: {input_ids_0.shape[1]}")
print(f"Q4: {total_token_positions}")
print(f"Q5: {q5_answer}")
print(f"Q6: {q6_answer}")
print(f"Q7: {trainable_params}")
print(f"Q8: {first_item_shape[0]}")
print(f"Q9: {q9_answer}")
print(f"Q10: {q10_answer}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 22.8 MB/s eta 0:00:0000:0100:01
Q1 - Encoded label at index 150: 2
Q2 - Length of formatted Option B input: 407
Q3 - input_ids shape: torch.Size([1, 5, 128]) | second dim: 5
Q4 - Batch input_ids shape: torch.Size([16, 5, 128]) | total token positions: 10240


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Skipping import of cpp extensions due to inco

Q5 - Logits shape: torch.Size([1, 5]) | logits per question: 5
Q6 - Loss tensor dimensions: 0 | loss value: 1.6391332149505615
Q7 - Trainable parameters: 295681
Q8 - First item input_ids shape: torch.Size([5, 128]) | num choices: 5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Q9 - Final global_step: 4
Q10 - Probability of Option E: 0.1995

--- SUMMARY ---
Q1: 2
Q2: 407
Q3: 5
Q4: 10240
Q5: 5
Q6: 0
Q7: 295681
Q8: 5
Q9: 4
Q10: 0.1995
